# 🎬 Colab-Comerza — PyMotion v3

Convierte un diseño (JSON) en video animado 9:16 con locución y música.
Usa **PyMotion** (FreeType+HarfBuzz) para textos profesionales,
PIL para efecto Ken Burns, y AudioMixer para mezcla de audio.

---
### ⚡ GPU + Run all
`Runtime → Change runtime type → T4 GPU` → `Runtime → Run all`

In [ ]:
# 1. SETUP
import os, sys, json, re, subprocess, shutil, wave
from pathlib import Path

REPO = "https://github.com/alexdechile/colab-comerza"
if not os.path.exists("colab-comerza"):
    !git clone {REPO}
%cd colab-comerza

# System deps for PyMotion (Cairo)
!apt-get update -qq && apt-get install -y -qq libcairo2-dev pkg-config 2>&1 | tail -1

# Python deps
%pip install pymotion-studio gtts Pillow numpy -q 2>&1 | tail -1

# Montserrat fonts
!wget -q "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat-Regular.ttf" -O Montserrat-Regular.ttf
!wget -q "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat-Bold.ttf" -O Montserrat-Bold.ttf
!wget -q "https://raw.githubusercontent.com/google/fonts/main/ofl/montserrat/Montserrat-ExtraBold.ttf" -O Montserrat-ExtraBold.ttf

for f in sorted(Path().glob("Montserrat*.ttf")):
    print(f"  {f.name}: {f.stat().st_size} bytes")
print("Setup listo")


In [ ]:
# 2. IMPORTS + VERSION CHECK
import sys
if sys.version_info < (3, 12):
    print("! PyMotion requiere Python 3.12+")
    print("  Instalando Python 3.12...")
    !apt-get install -y -qq python3.12 python3.12-dev python3.12-venv 2>&1 | tail -3
    !python3.12 -m pip install pymotion-studio gtts Pillow numpy -q 2>&1 | tail -1
    print("  Python 3.12 listo, reinicia el runtime y ejecuta de nuevo.")
    sys.exit(0)

import numpy as np
from PIL import Image
from gtts import gTTS
import pymotion as pm

print("Colab-Comerza v3 (PyMotion)")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO")
print(f"Python {sys.version}")


In [ ]:
# 3. CARGAR DISEÑO
with open("mi-diseno.json") as f:
    design = json.load(f)

CANVAS_W = design['canvasSize']['width']
CANVAS_H = design['canvasSize']['height']
OUT_W, OUT_H = 540, 960
SCALE = OUT_W / CANVAS_W
FPS = 24
MAX_DUR = 14.0
TOTAL_FRAMES = int(MAX_DUR * FPS)

def strip_html(t):
    return re.sub(r'<[^>]+>', '', t).strip()

text_elements = []
img_info = None

for el in design['elements']:
    if el['type'] == 'image':
        img_info = {
            'x': el['x'] * SCALE,
            'y': el['y'] * SCALE,
            'w': el['width'] * SCALE,
            'h': el['height'] * SCALE,
        }
        print(f"  Imagen: ({int(img_info['x'])},{int(img_info['y'])}) {int(img_info['w'])}x{int(img_info['h'])}")
    elif el['type'] == 'text':
        content = strip_html(el['content'])
        if not content:
            continue
        text_elements.append({
            'content': content,
            'x': el['x'] * SCALE,
            'y': el['y'] * SCALE,
            'size': el['fontSize'] * SCALE,
            'color': el['color'],
            'shadow': el.get('hasShadow', False),
            'sox': el.get('shadowOffsetX', 0) * SCALE,
            'soy': el.get('shadowOffsetY', 0) * SCALE,
            'sc': el.get('shadowColor', 'rgba(0,0,0,0.5)'),
            'align': el.get('alignment', 'left'),
            'width': el.get('width', CANVAS_W) * SCALE,
            'weight': el.get('fontWeight', 400),
        })
        print(f"  Texto: '{content}' @ ({int(text_elements[-1]['x'])},{int(text_elements[-1]['y'])})")

print(f"Total: {len(text_elements)} textos, escala={SCALE:.2f}")


In [ ]:
# 4. GENERAR FRAMES KEN BURNS (PIL)
print("Generando Ken Burns...")

bg_src = Image.open("imagen.png").convert("RGBA")

img_x = int(img_info['x'])
img_y = int(img_info['y'])
img_w = int(img_info['w'])
img_h = int(img_info['h'])

# Ken Burns: slow zoom-in with slight upward pan
ZOOM_START, ZOOM_END = 1.0, 1.20
PAN_X_START, PAN_X_END = 0, 15
PAN_Y_START, PAN_Y_END = 0, -12

os.makedirs("frames_kenburns", exist_ok=True)

for fi in range(TOTAL_FRAMES):
    t = fi / TOTAL_FRAMES
    # ease-in-out quadratic
    if t < 0.5:
        ease_t = 2 * t * t
    else:
        ease_t = 1 - (-2 * t + 2) ** 2 / 2

    zoom = ZOOM_START + (ZOOM_END - ZOOM_START) * ease_t
    pan_x = PAN_X_START + (PAN_X_END - PAN_X_START) * ease_t
    pan_y = PAN_Y_START + (PAN_Y_END - PAN_Y_START) * ease_t

    frame_w = int(img_w * zoom)
    frame_h = int(img_h * zoom)
    resized = bg_src.resize((frame_w, frame_h), Image.LANCZOS)

    cx = img_x + (img_w - frame_w) // 2 + int(pan_x)
    cy = img_y + (img_h - frame_h) // 2 + int(pan_y)

    frame = Image.new("RGBA", (OUT_W, OUT_H), (255, 255, 255, 255))
    frame.paste(resized, (cx, cy), resized)
    frame.convert("RGB").save(f"frames_kenburns/frame_{fi:04d}.png")

    if fi % 60 == 0 or fi == TOTAL_FRAMES - 1:
        print(f"  frame {fi+1}/{TOTAL_FRAMES}")

print("Ken Burns frames listos")


In [ ]:
# 5. ARMAR VIDEO KEN BURNS CON FFMPEG
print("Creando video Ken Burns...")
KENBURNS_VIDEO = "kenburns_temp.mp4"

!ffmpeg -y -framerate {FPS} -i frames_kenburns/frame_%04d.png \
  -c:v libx264 -pix_fmt yuv420p -preset ultrafast -crf 22 \
  {KENBURNS_VIDEO} 2>&1 | tail -3

print(f"Video: {KENBURNS_VIDEO} ({os.path.getsize(KENBURNS_VIDEO)//1024} KB)")


In [ ]:
# 6. TTS (gTTS)
print("Generando locucion...")
tts_files = []
timing_starts = [0.0, 3.0]

for i, te in enumerate(text_elements):
    text = te['content']
    print(f"  TTS: {text}")
    tts = gTTS(text, lang="es", tld="com.mx", slow=False)
    path = f"tts_{i}.mp3"
    tts.save(path)
    start = timing_starts[i] if i < len(timing_starts) else (tts_files[-1]['end'] + 0.5)
    tts_files.append({'text': text, 'start': start, 'path': path})

# Get durations via ffprobe
for i, tf in enumerate(tts_files):
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", tf['path']],
        capture_output=True, text=True,
    )
    dur = float(result.stdout.strip())
    tf['dur'] = dur
    tf['end'] = tf['start'] + dur
    print(f"  '{tf['text']}': {dur:.2f}s @ {tf['start']:.1f}s")

total_dur = min(max(MAX_DUR, max(t['end'] for t in tts_files) + 0.5), MAX_DUR)
total_frames = int(total_dur * FPS)
print(f"Duracion: {total_dur:.1f}s = {total_frames} frames")


In [ ]:
# 7. MUSICA DE FONDO (C-Am-F-G con bateria)
print("Generando musica (C-Am-F-G)...")
SR = 44100
n = int(total_dur * SR)
t = np.linspace(0, total_dur, n, endpoint=False)

section_len = 3.5
section_idx = (t // section_len).astype(int) % 4

chords = [
    (261.63, 329.63, 392.00),
    (220.00, 261.63, 329.63),
    (174.61, 220.00, 261.63),
    (196.00, 246.94, 293.66),
]
roots = [261.63, 220.00, 174.61, 196.00]

pad = np.zeros(n)
for sec in range(4):
    mask = (section_idx == sec)
    f1, f2, f3 = chords[sec]
    pad[mask] = (np.sin(2*np.pi*f1*t[mask]) + np.sin(2*np.pi*f2*t[mask]) + np.sin(2*np.pi*f3*t[mask])) / 3
pad *= 0.35

bass = np.zeros(n)
for sec in range(4):
    mask = (section_idx == sec)
    bass[mask] = np.sin(2*np.pi*roots[sec]*t[mask]) * 0.2

kick = np.zeros(n)
for b in range(0, n, int(SR * 0.5)):
    end = min(b+1200, n)
    seg = np.linspace(0, (end-b)/SR, end-b, endpoint=False)
    kick[b:end] = 0.6 * np.sin(2*np.pi*55*seg) * np.exp(-seg*25)

hh = np.zeros(n)
np.random.seed(42)
for b in range(int(SR*0.25), n, int(SR * 0.5)):
    end = min(b+600, n)
    seg = np.linspace(0, (end-b)/SR, end-b, endpoint=False)
    hh[b:end] = 0.3 * np.random.randn(end-b) * np.exp(-seg*40)

music_arr = np.clip(pad + bass + kick + hh, -0.9, 0.9) * 0.5
music_arr_int16 = (music_arr * 32767).astype(np.int16)

with wave.open("music_bg.wav", "w") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(SR)
    wf.writeframes(music_arr_int16.tobytes())

print(f"Musica lista ({n} samples)")


In [ ]:
# 8. COMPOSICION PYMOTION
print("Armando composicion PyMotion...")

comp = pm.Composition(width=OUT_W, height=OUT_H, fps=FPS, duration=total_frames)

# --- Track 1: Fondo Ken Burns (VideoClip) ---
bg_track = pm.Track(name="kenburns")
kenburns_video = pm.VideoClip(KENBURNS_VIDEO)
kenburns_video.set_duration(total_frames)
bg_track.add(kenburns_video)
comp.add_track(bg_track)

# --- Tracks 2+: Textos con animacion ---
FONT_PATHS = {
    800: "Montserrat-ExtraBold.ttf",
    700: "Montserrat-Bold.ttf",
    400: "Montserrat-Regular.ttf",
}

def get_font_path(weight):
    for w in [800, 700, 400]:
        if weight >= w:
            return FONT_PATHS[w]
    return "Montserrat-Regular.ttf"

for ti, te in enumerate(text_elements):
    track = pm.Track(name=f"text_{ti}")
    td = tts_files[ti]

    start_frame = int(td['start'] * FPS)
    end_frame = int(td['end'] * FPS)
    dur_frames = end_frame - start_frame

    color_hex = te['color']
    if not color_hex.startswith('#'):
        color_hex = '#121645'

    shadow_obj = None
    if te['shadow']:
        shadow_obj = pm.Shadow(
            color=pm.Color(0, 0, 0, 0.5),
            offset_x=te['sox'],
            offset_y=te['soy'],
            blur=4.0,
        )

    align = te['align']

    font_path = get_font_path(te['weight'])
    txt = pm.TextClip(
        te['content'],
        font=font_path,
        size=te['size'],
        color=color_hex,
        shadow=shadow_obj,
        align=align,
        max_width=te['width'],
    )

    # Posicion X: centro del contenedor si align=center
    if align == 'center':
        px = te['x'] + te['width'] / 2
    else:
        px = te['x']
    py = te['y']
    txt.set_position(px, py)
    txt.set_duration(dur_frames)
    txt.at(start_frame)

    track.add(txt)
    comp.add_track(track)

    print(f"  Texto {ti}: '{te['content'][:30]}...' @ ({int(px)},{int(py)}) size={te['size']:.0f}px")

print("Composicion lista")


In [ ]:
# 9. RENDER VIDEO (sin audio)
print("Renderizando video PyMotion...")
VIDEO_OUT = "comerza_sin_audio.mp4"
comp.render(VIDEO_OUT, preset="h264_fast")
print(f"Video renderizado: {VIDEO_OUT} ({os.path.getsize(VIDEO_OUT)//1024} KB)")


In [ ]:
# 10. MEZCLA DE AUDIO
print("Mezclando audio...")

SR_AUDIO = 44100
mixer = pm.AudioMixer(sample_rate=SR_AUDIO, channels=2)

# Cargar musica con numpy desde WAV
with wave.open("music_bg.wav", "r") as wf:
    frames = wf.readframes(wf.getnframes())
    music_mono = np.frombuffer(frames, dtype=np.int16).astype(np.float64) / 32767.0
music_stereo = np.column_stack([music_mono, music_mono])

mixer.add(
    pm.AudioClipData(samples=music_stereo, start_sample=0),
    track="music",
)
mixer.set_volume("music", 0.45)

# Cargar TTS con numpy desde MP3 vía ffmpeg
for i, tf in enumerate(tts_files):
    # Decode MP3 to WAV with ffmpeg
    wav_path = f"tts_{i}.wav"
    subprocess.run([
        "ffmpeg", "-y", "-i", tf['path'],
        "-acodec", "pcm_s16le", "-ar", str(SR_AUDIO),
        "-ac", "1", wav_path,
    ], capture_output=True)

    with wave.open(wav_path, "r") as wf:
        frames = wf.readframes(wf.getnframes())
        tts_mono = np.frombuffer(frames, dtype=np.int16).astype(np.float64) / 32767.0
    tts_stereo = np.column_stack([tts_mono, tts_mono])

    start_sample = int(tf['start'] * SR_AUDIO)
    mixer.add(
        pm.AudioClipData(samples=tts_stereo, start_sample=start_sample),
        track=f"tts_{i}",
    )
    mixer.set_volume(f"tts_{i}", 1.0)

    os.remove(wav_path)

# Render mix
mixed = mixer.render()

# Guardar como WAV
with wave.open("audio_final.wav", "w") as wf:
    wf.setnchannels(2)
    wf.setsampwidth(2)
    wf.setframerate(SR_AUDIO)
    wf.writeframes(mixed.astype(np.int16).tobytes())

print(f"Audio mezclado: {len(mixed)} samples")


In [ ]:
# 11. COMBINAR VIDEO + AUDIO
print("Combinando video + audio...")
OUTPUT = "comerza_animado.mp4"

!ffmpeg -y -i {VIDEO_OUT} -i audio_final.wav \
  -c:v copy -c:a aac -b:a 192k -shortest \
  {OUTPUT} 2>&1 | tail -3

file_size_kb = os.path.getsize(OUTPUT) // 1024
print(f"\nVideo final: {OUTPUT}")
print(f"Tama\u00f1o: {file_size_kb} KB")


In [ ]:
# 12. LIMPIEZA
print("Limpiando archivos temporales...")
shutil.rmtree("frames_kenburns", ignore_errors=True)
os.makedirs("frames_kenburns", exist_ok=True)
with open("frames_kenburns/.gitkeep", "w") as f:
    f.write("")
for f in ["music_bg.wav", "audio_final.wav", VIDEO_OUT]:
    if os.path.exists(f):
        os.remove(f)
print("Listo")


In [ ]:
# 13. DESCARGAR
from google.colab import files
files.download(OUTPUT)
